In [7]:
import numpy as np
import scipy.ndimage
import ast
from pathlib import Path
from tqdm import tqdm
import sys
sys.path.append(r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\10_code\UTvsXCT-preprocessing')
sys.path.append(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/UTvsXCT-preprocessing')
import dbtools.dbtools as db
import dbtools.load as load
from preprocess_tools import io

# Database connection

In [8]:
try:
    conn = db.connect()
    print('Connected to the database')
except Exception as error:
    print(error)

Connected to the database


# Parameters

In [9]:
original_paths = [
    Path(r"/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_09_5/volume_eq_aligned.tif"),
    Path(r"/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned.tif"),
    Path(r"/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_2/volume_eq_aligned.tif"),
    Path(r"/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_3/volume_eq_aligned.tif")
]

target_resolution = 0.025   # mm/voxel — must be coarser than the original (downscale only)

save_to_database = True

# Data retrieval

In [10]:
UNC_DATA_ROOT   = r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente'
LOCAL_DATA_ROOT = Path(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente')

def normalize_path(path_str):
    # Normalize a DB path to the local Linux mount point.
    # DB paths may be Windows UNC or Linux depending on which machine saved the measurement.
    p = str(path_str).strip().replace('\\', '/')
    unc_norm = UNC_DATA_ROOT.replace('\\', '/')
    if p.lower().startswith(unc_norm.lower()):
        relative = p[len(unc_norm):].lstrip('/')
        return str(LOCAL_DATA_ROOT / relative)
    return p

measurements_table      = db.relation_metadata('measurements', 'samples', 'sample_measurements')
measurements_meta_table = db.get_data_metadata('measurements')
measurementtypes_table  = db.get_data_metadata('measurementtypes')

measurements_table['file_path_normalized'] = (
    measurements_table['file_path_measurement'].apply(normalize_path)
)

In [11]:
# Metadata keys handled as explicit load_xct_measurement parameters — do not copy via additional_metadata
SKIP_METADATA_KEYS = {
    'height', 'width', 'depth', 'dtype', 'file_type',
    'aligned', 'equalized', 'axes_order', 'transformations',
    'actual_voxel_size',
}
# Columns from the main measurements table (not from measurement_metadata)
MAIN_TABLE_COLS = {'id', 'file_path', 'measurementtype_id', 'parent_measurement_id'}

original_ids                  = []
original_db_paths             = []
original_measurementtype_ids  = []
original_sample_names         = []
original_resolutions          = []
original_aligned              = []
original_equalized            = []
original_axes_orders          = []
original_additional_metadatas = []

for original_path in original_paths:
    path_str = str(original_path)

    # Match against the normalised column so UNC-stored paths are found from a Linux input
    row = measurements_table.loc[measurements_table['file_path_normalized'] == path_str]
    assert not row.empty, f"Measurement not found in database: {path_str}"

    original_id        = int(row['id_measurement'].values[0])
    measurementtype_id = int(row['measurementtype_id_measurement'].values[0])
    sample_names       = list(row['name_sample'].values)
    db_path            = row['file_path_measurement'].values[0]

    # Metadata — stored as 'value type' strings
    meta_row = measurements_meta_table[
        measurements_meta_table['id_measurement'] == original_id
    ]

    def _parse_bool(col):
        if col not in meta_row.columns or meta_row[col].isna().all():
            return False
        return meta_row[col].values[0].split()[0] == 'True'

    def _parse_list(col):
        if col not in meta_row.columns or meta_row[col].isna().all():
            return ['z', 'y', 'x']
        raw = meta_row[col].values[0].rsplit(' ', 1)[0]
        return ast.literal_eval(raw)

    # Resolution: prefer actual_voxel_size from measurement metadata, fall back to measurementtype
    if ('actual_voxel_size_measurement' in meta_row.columns
            and not meta_row['actual_voxel_size_measurement'].isna().all()):
        original_resolution = float(
            meta_row['actual_voxel_size_measurement'].values[0].split(' ')[0]
        )
        resolution_source = 'actual_voxel_size (measurement metadata)'
    else:
        mt_row = measurementtypes_table[
            measurementtypes_table['id_measurementtype'] == measurementtype_id
        ]
        original_resolution = float(
            mt_row['voxel_size_measurementtype'].values[0].split(' ')[0]
        )
        resolution_source = 'voxel_size (measurementtype)'

    aligned    = _parse_bool('aligned_measurement')
    equalized  = _parse_bool('equalized_measurement')
    axes_order = _parse_list('axes_order_measurement')

    # Collect every other metadata field to carry over to the rescaled measurement
    additional_metadata = []
    for col in meta_row.columns:
        if not col.endswith('_measurement'):
            continue
        key = col[:-len('_measurement')]
        if key in MAIN_TABLE_COLS or key in SKIP_METADATA_KEYS:
            continue
        if meta_row[col].isna().all():
            continue
        cell_value = str(meta_row[col].values[0])
        value, type_ = cell_value.rsplit(' ', 1)
        additional_metadata.append({'key': key, 'value': value, 'type': type_})
    additional_metadata = additional_metadata if additional_metadata else None

    original_ids.append(original_id)
    original_db_paths.append(db_path)
    original_measurementtype_ids.append(measurementtype_id)
    original_sample_names.append(sample_names)
    original_resolutions.append(original_resolution)
    original_aligned.append(aligned)
    original_equalized.append(equalized)
    original_axes_orders.append(axes_order)
    original_additional_metadatas.append(additional_metadata)

    zoom_factor = original_resolution / target_resolution
    extra_keys = [m['key'] for m in additional_metadata] if additional_metadata else []
    print(
        f"{path_str}\n"
        f"  ID: {original_id}  MeasurementType: {measurementtype_id}\n"
        f"  DB path: {db_path}\n"
        f"  Original resolution: {original_resolution} mm/voxel [{resolution_source}]\n"
        f"  Target: {target_resolution} mm/voxel  Zoom: {zoom_factor:.4f}\n"
        f"  Samples: {sample_names}  Aligned: {aligned}  Equalized: {equalized}\n"
        f"  Extra metadata to copy: {extra_keys}\n"
    )

/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_09_5/volume_eq_aligned.tif
  ID: 527  MeasurementType: 6
  DB path: \\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\02_XCT_data\Fabricacion Nacho\05_Probetas_Nacho_2025\probetas\Na_09_5\volume_eq_aligned.tif
  Original resolution: 0.0206 mm/voxel [actual_voxel_size (measurement metadata)]
  Target: 0.025 mm/voxel  Zoom: 0.8240
  Samples: ['Na_09_5']  Aligned: True  Equalized: True
  Extra metadata to copy: []

/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned.tif
  ID: 528  MeasurementType: 6
  DB path: \\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\02_XCT_data\Fabricacion Nacho\05_Probetas_Nacho_2025\probetas\Na_10_1\volume_eq_aligned.tif
  Original resolution: 0.0206 mm/voxel [actual_voxel_size (measurement metadata)]
  Target: 0.025 mm/voxel  Zoom: 0.8240
  Samples

# Rescaling

**Only downscaling is supported** (`target_resolution` must be coarser than the original).  
An assertion is raised for any measurement where this does not hold.

In [12]:
for i, original_path in enumerate(tqdm(original_paths)):

    assert target_resolution > original_resolutions[i], (
        f"target_resolution ({target_resolution}) must be larger than the original "
        f"({original_resolutions[i]}) — this notebook only downscales."
    )

    volume = io.load_tif(original_path)

    zoom_factor = original_resolutions[i] / target_resolution  # always < 1
    rescaled = scipy.ndimage.zoom(volume, zoom_factor, order=1)
    rescaled = rescaled.astype(volume.dtype)

    save_path = original_path.parent / (original_path.stem + '_rescaled' + original_path.suffix)
    io.save_tif(save_path, rescaled)

    h, w, d = rescaled.shape
    transformation = (
        f"Downscaled from {original_resolutions[i]} mm/voxel to {target_resolution} mm/voxel "
        f"using scipy.ndimage.zoom (order=1, zoom_factor={zoom_factor:.4f}). "
        f"Original shape: {volume.shape}, rescaled shape: {rescaled.shape}. "
        f"Done with UTvsXCT-PREPROCESSING toolkit 0.1.23."
    )

    if save_to_database:
        load.load_xct_measurement(
            conn,
            str(save_path),
            original_measurementtype_ids[i],
            h, w, d,
            str(rescaled.dtype),
            'tif',
            original_sample_names[i],
            original_aligned[i],
            original_equalized[i],
            original_axes_orders[i],
            parent_measurement_path=original_db_paths[i],
            transformations=transformation,
            additional_metadata=original_additional_metadatas[i],
        )
        print(f"Saved and registered in database: {save_path}")
    else:
        print(f"[dry run] {save_path}  shape={rescaled.shape}")

 25%|██▌       | 1/4 [00:28<01:25, 28.61s/it]

XCT measurement from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_09_5/volume_eq_aligned_rescaled.tif' loaded with ID: 606
Saved and registered in database: /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_09_5/volume_eq_aligned_rescaled.tif


 50%|█████     | 2/4 [00:57<00:57, 28.59s/it]

XCT measurement from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned_rescaled.tif' loaded with ID: 607
Saved and registered in database: /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned_rescaled.tif


 75%|███████▌  | 3/4 [01:27<00:29, 29.38s/it]

XCT measurement from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_2/volume_eq_aligned_rescaled.tif' loaded with ID: 608
Saved and registered in database: /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_2/volume_eq_aligned_rescaled.tif


100%|██████████| 4/4 [01:56<00:00, 29.12s/it]

XCT measurement from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_3/volume_eq_aligned_rescaled.tif' loaded with ID: 609
Saved and registered in database: /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_3/volume_eq_aligned_rescaled.tif
